# Experiment 6 — Building Hybrid Quantum-Classical Models (QAOA)
Solves **MaxCut** and **TSP** using QAOA — quantum circuit for evaluation, classical optimizer for parameter tuning.

In [ ]:
!pip install qiskit qiskit-aer qiskit-algorithms qiskit-optimization matplotlib numpy scipy -q

---
## Part A — MaxCut with QAOA

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit_aer import Aer
from itertools import combinations

## Step 1 — Define the Graph

In [ ]:
num_nodes = 5

edge_list = [
    (0, 1), (0, 2),
    (1, 3),
    (3, 4),
    (4, 2),
    (2, 1)
]

gamma = Parameter('gamma')
beta  = Parameter('beta')

print(f"Graph: {num_nodes} nodes, {len(edge_list)} edges")
print(f"Edges: {edge_list}")

## Step 2 — Build QAOA Circuit (p=1 layer)

In [ ]:
def build_qaoa_circuit():
    circuit = QuantumCircuit(num_nodes)

    for node in range(num_nodes):
        circuit.h(node)

    for node_i, node_j in edge_list:
        circuit.cx(node_i, node_j)
        circuit.rz(2 * gamma, node_j)
        circuit.cx(node_i, node_j)

    for node in range(num_nodes):
        circuit.rx(2 * beta, node)

    return circuit

qaoa_circuit = build_qaoa_circuit()
print("QAOA Circuit:")
print(qaoa_circuit.draw('text'))

## Step 3 — Assign Parameters and Run

In [ ]:
gamma_value = 0.8
beta_value  = 0.7

bound_circuit = qaoa_circuit.assign_parameters({gamma: gamma_value, beta: beta_value})
bound_circuit.measure_all()

backend        = Aer.get_backend('qasm_simulator')
compiled       = transpile(bound_circuit, backend)
job            = backend.run(compiled, shots=2048)
measurement    = job.result().get_counts()

print("Measurement Counts:")
print(measurement)

## Step 4 — Plot Output Distribution

In [ ]:
total_shots   = sum(measurement.values())
probabilities = {k: v / total_shots for k, v in measurement.items()}

plt.figure(figsize=(10, 4))
plt.bar(list(probabilities.keys()), list(probabilities.values()))
plt.xticks(rotation=90)
plt.xlabel("Bitstrings")
plt.ylabel("Probability")
plt.title("QAOA Output Distribution — MaxCut")
plt.tight_layout()
plt.show()

## Step 5 — Compute Cut Value and Find Best Partition

In [ ]:
def count_cut(bitstring):
    cut  = 0
    bits = [int(b) for b in bitstring[::-1]]
    for node_i, node_j in edge_list:
        if bits[node_i] != bits[node_j]:
            cut += 1
    return cut

print("Bitstring → Cut Value:")
for bitstring in sorted(measurement, key=count_cut, reverse=True)[:10]:
    print(f"  {bitstring}  →  Cut = {count_cut(bitstring)}  (seen {measurement[bitstring]} times)")

best_partition = max(measurement, key=count_cut)
print(f"\nBest partition found: {best_partition}")
print(f"Cut value: {count_cut(best_partition)}")

---
## Part B — TSP with QAOA

## Imports

In [ ]:
import scipy.sparse as sp
if not hasattr(sp.csr_matrix, "H"):
    sp.csr_matrix.H = property(lambda self: self.getH())

from qiskit.primitives import Sampler
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization.applications import Tsp
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_optimization.algorithms import MinimumEigenOptimizer

## Step 1 — Define Distance Matrix (4 Cities)

In [ ]:
city_distances = np.array([
    [0,  10, 15, 20],
    [10,  0, 35, 25],
    [15, 35,  0, 30],
    [20, 25, 30,  0]
])

print("Distance Matrix (city i to city j):")
print(city_distances)

## Step 2 — Convert TSP to QUBO

In [ ]:
tsp_problem    = Tsp(city_distances)
quadratic_prog = tsp_problem.to_quadratic_program()

qubo = QuadraticProgramToQubo().convert(quadratic_prog)
print("TSP converted to QUBO successfully.")

## Step 3 — Set Up QAOA Solver

In [ ]:
optimizer     = COBYLA(maxiter=100)
sampler       = Sampler()
qaoa_solver   = QAOA(sampler=sampler, optimizer=optimizer, reps=2)
tsp_optimizer = MinimumEigenOptimizer(qaoa_solver)

## Step 4 — Solve with QAOA

In [ ]:
print("Running QAOA for TSP...")
qaoa_result = tsp_optimizer.solve(qubo)

print("\n=== QAOA Solution ===")
print("Binary:", qaoa_result.x)
print("Cost:  ", qaoa_result.fval)

qaoa_route = tsp_problem.interpret(qaoa_result.x)
print("Route: ", qaoa_route)

## Step 5 — Classical Exact Solution for Comparison

In [ ]:
exact_optimizer = MinimumEigenOptimizer(NumPyMinimumEigensolver())
exact_result    = exact_optimizer.solve(qubo)

print("\n=== Exact Classical Solution ===")
print("Binary:", exact_result.x)
print("Cost:  ", exact_result.fval)

exact_route = tsp_problem.interpret(exact_result.x)
print("Route: ", exact_route)

print(f"\nQAOA Cost  : {qaoa_result.fval}")
print(f"Exact Cost : {exact_result.fval}")